In [1]:
# !python -m pip install -U python-woc pandas matplotlib
# please clear the output of this cell in your notebook before checking it in

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from woc.remote import WocMapsRemote 
from tqdm import tqdm
import pandas as pd
# creates the client
woc = WocMapsRemote( base_url="https://worldofcode.org/api/")
# if you got an API key
# woc = WocMapsRemote( base_url="https://worldofcode.org/api/", api_key="woc-XXXXXX-YYYYYY" )

# Now for each of the ten projects you were assigned get commits, e.g.
# projects = ['mmorise/World']
projects = [
    "aidenlab/straw",
    "corybrunson/ggalluvial",
    "snowleopard/alga",
    "maximecb/gym-minigrid",
    "seung-lab/cloud-volume",
    "STVIR/pysot",
    "google-research/torchsde",
    "tidymodels/rsample",
    "coin-or/Ipopt",
    "OHDSI/PatientLevelPrediction",
]
list_df_commits = []
index = 0
for prj in projects:
  # Convert GitHub repository names to WoC V2412 project names
  prj = prj.lower().replace('/','_',2)
  # print(f"Retrieving: {prj}", flush=True)
  commits = woc.get_values('p2c', prj)
  df = pd.DataFrame(commits, columns=["sha1"])
  df['project'] = prj
  list_df_commits.append(df)
df = pd.concat(list_df_commits, ignore_index=True)
#perhaps save the list (if no errors) so you do not need to retrieve them again
df.to_csv('df_commits.csv')
df.head(10)

,sha1,project
0,014d06ce8337aab2b4d7e9da90d2d0d22537d118,aidenlab_straw
1,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,aidenlab_straw
2,0397a5027626eab94a326650cb7bf8bf73e6084c,aidenlab_straw
3,03cf1d6674f4d3eeed208f55bd5e31f74b381151,aidenlab_straw
4,03fbfdc1d863e8aed6ce70bee27c54daab2d388b,aidenlab_straw
5,043329007b29d5b1cd79da01ba6f72a27606bd7d,aidenlab_straw
6,043684bef67e20f1e3543063be76830e4c299c78,aidenlab_straw
7,0542e0fdbac861f172b286a3ddd98248fb4b830f,aidenlab_straw
8,055d71f3203cd22026296d4e0a99506b356a4c78,aidenlab_straw
9,05f413f4207fd01abed54155563eb43d3c582bdb,aidenlab_straw


In [3]:
# If the number of commits its not very large, you can try to get them all at the same time,
# The max batch size is 10
import time
# let us first split df['sha1'] in chunks
chunks = [df['sha1'][x:x+10] for x in range(0, len(df), 10)]
commit_data = []

for chunk in tqdm(chunks): # iterate over the commits
  # res, err = woc.show_content_many('commit',chunk.to_list())

  # commit.tch returns the same data but faster
  res, err = woc.get_values_many('commit.tch',chunk.to_list())
  res = {k: v[0] for k, v in res.items()}  # this conversion is necessary because of the internal implementation

  # to walk around the rate limit
  time.sleep(1)

  if err: # check for errors
    print('Got Errors', err)

  for commit_sha, commit in res.items():
    # flatten commit objects
    commit_data.append({
          'commit': commit_sha,
          'tree': commit[0],
          'parent': list(commit[1]),
          'author': commit[2][0],
          'author_time': int(commit[2][1]),
          'author_tz': commit[2][2],
          'committer': commit[3][0],
          'committer_time': int(commit[3][1]),
          'committer_tz': commit[3][2],
          'message': commit[4],
    })

df_commit_data = pd.DataFrame(commit_data)
df_commit_data = df_commit_data.merge(df, left_on='commit', right_on='sha1')
df_commit_data.to_csv('df_commit_data.csv', index=False)
df_commit_data.head(2)

  0%|          | 0/2274 [00:00<?, ?it/s]

  0%|          | 2/2274 [00:02<38:29,  1.02s/it]

Got Errors {'0686ed71f060208739b1da33c809ec7f73e48ff5': 'Key 0686ed71f060208739b1da33c809ec7f73e48ff5 not found in /da5_fast/All.sha1c/commit_6.tch'}


  1%|          | 12/2274 [00:12<38:18,  1.02s/it]

Got Errors {'4247b698dfcb8dde46198761994c3a94f00f2a06': 'Key 4247b698dfcb8dde46198761994c3a94f00f2a06 not found in /da5_fast/All.sha1c/commit_66.tch'}


...

Got Errors {'fd4fe5a61b8fce59f616073a3110f48eb145cf78': 'Key fd4fe5a61b8fce59f616073a3110f48eb145cf78 not found in /da5_fast/All.sha1c/commit_125.tch'}


 40%|████      | 911/2274 [15:40<23:28,  1.03s/it]

Got Errors {'ff85bc8665736f293cd03d38b0db0bc3c16b9b23': 'Key ff85bc8665736f293cd03d38b0db0bc3c16b9b23 not found in /da5_fast/All.sha1c/commit_127.tch'}


100%|██████████| 2274/2274 [39:03<00:00,  1.03s/it]


,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,014d06ce8337aab2b4d7e9da90d2d0d22537d118,47538609f59c6da7ae35aec116b1ce81607a436d,[909b3cedd5d3756829cd03fcb2caa925a909a090],Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,-0400,Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,-0400,parallelize getting blocks\n,014d06ce8337aab2b4d7e9da90d2d0d22537d118,aidenlab_straw
1,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,0d917067d4beba5b9beb6fc93c09f6e6c26ed315,"[a69b35326e0568b3114be921d352780c8eab5251, 409...",Muhammad Saad Shamim <sa501428@gmail.com>,1643839632,-0600,GitHub <noreply@github.com>,1643839632,-0600,Merge pull request #91 from sa501428/helper3\n...,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,aidenlab_straw


In [23]:
# what does the first record looks like?
# commit  parents
commit_data = pd.read_csv("df_commit_data.csv")
# print(commit_data[1])
df_commit_data = pd.DataFrame(commit_data)
# v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
# print(v)
# print(commit_data)
df_commit_data.head(10)

,commit,tree,parent,author,author_time,author_tz,committer,committer_time,committer_tz,message,sha1,project
0,014d06ce8337aab2b4d7e9da90d2d0d22537d118,47538609f59c6da7ae35aec116b1ce81607a436d,['909b3cedd5d3756829cd03fcb2caa925a909a090'],Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,-400,Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,-400,parallelize getting blocks\n,014d06ce8337aab2b4d7e9da90d2d0d22537d118,aidenlab_straw
1,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,0d917067d4beba5b9beb6fc93c09f6e6c26ed315,"['a69b35326e0568b3114be921d352780c8eab5251', '...",Muhammad Saad Shamim <sa501428@gmail.com>,1643839632,-600,GitHub <noreply@github.com>,1643839632,-600,Merge pull request #91 from sa501428/helper3\n...,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,aidenlab_straw
2,0397a5027626eab94a326650cb7bf8bf73e6084c,456d17648ba913c458553d083cdd48308d5cc0b9,['fefd0476927b064b004e5df0eb83bf1db717b6e7'],nchernia <nchernia@gmail.com>,1476365702,200,nchernia <nchernia@gmail.com>,1476365702,200,updated README\n,0397a5027626eab94a326650cb7bf8bf73e6084c,aidenlab_straw
3,03cf1d6674f4d3eeed208f55bd5e31f74b381151,24dec8256fdc1e23a175b012100725e23290aeb6,['b2cbe5664bc878d6bf67f9cb82db2101bba0423d'],Muhammad S Shamim <sa501428@gmail.com>,1644528519,-600,Muhammad S Shamim <sa501428@gmail.com>,1644528519,-600,new usage\n,03cf1d6674f4d3eeed208f55bd5e31f74b381151,aidenlab_straw
4,03fbfdc1d863e8aed6ce70bee27c54daab2d388b,310b9d52339e5e5e301189c44ed28f360015b6f5,['c92b9167129e6c56e9eb02edc54d6f726b756a62'],Dimos Goundaroulis <dgound@gmail.com>,1632171347,-500,Dimos Goundaroulis <dgound@gmail.com>,1632171347,-500,published v1.0.0.1 to pypi.org\n,03fbfdc1d863e8aed6ce70bee27c54daab2d388b,aidenlab_straw
5,043329007b29d5b1cd79da01ba6f72a27606bd7d,e548ebd32694d6a6ea6174e63d12253fb41f55d7,"['fc1ddf5978b577caec992809b4249a89815465d0', '...",Muhammad Saad Shamim <sa501428@gmail.com>,1743906200,-500,GitHub <noreply@github.com>,1743906200,-500,Merge branch 'master' into temp_merge,043329007b29d5b1cd79da01ba6f72a27606bd7d,aidenlab_straw
6,043684bef67e20f1e3543063be76830e4c299c78,9756c0b3d14ae3e5efc4795fa3901854e86bdcae,"['a015f83114788d7ab97e0338c29c14b92e768d86', '...",Muhammad Saad Shamim <sa501428@gmail.com>,1621030440,-500,Muhammad Saad Shamim <sa501428@gmail.com>,1621030440,-500,Merge branch 'aidenlab:master' into master\n,043684bef67e20f1e3543063be76830e4c299c78,aidenlab_straw
7,0542e0fdbac861f172b286a3ddd98248fb4b830f,bf5c8c890bf06e2d64423f137cc3c25484f783e1,['85045e5092a4858edddf027be1010a084e48921b'],nchernia <nchernia@gmail.com>,1571860390,-400,nchernia <nchernia@gmail.com>,1571860390,-400,updating since norm can be arbitrary\n,0542e0fdbac861f172b286a3ddd98248fb4b830f,aidenlab_straw
8,055d71f3203cd22026296d4e0a99506b356a4c78,d613de3f99c5c12d8d3fc513ec61f45e0f1ca4a5,['61d1d891cc1bcc26ff2bad060db1d68b52de8369'],nchernia <nchernia@gmail.com>,1579892580,-500,nchernia <nchernia@gmail.com>,1579892580,-500,changes to be consistent with pybind version\n,055d71f3203cd22026296d4e0a99506b356a4c78,aidenlab_straw
9,05f413f4207fd01abed54155563eb43d3c582bdb,d004be092e75420fe702ae18ca9bd85f092b5280,['61d1d891cc1bcc26ff2bad060db1d68b52de8369'],nchernia <neva@broadinstitute.org>,1579891694,-500,GitHub <noreply@github.com>,1579891694,-500,Update README.md,05f413f4207fd01abed54155563eb43d3c582bdb,aidenlab_straw


In [9]:
# Now that the data is retrieved, save it and commit to your GH fork
# First, we need to flatten info in order to export as csv
# our dataframe will have columns project,'commit, author, time, message'
dfinf = pd.DataFrame(columns=['project', 'commit', 'author', 'time', 'message'])
for k in commit_data:
  row = pd.Series({'project':prj, 'commit': k['commit'], 'author': k['author'], 'time': k['author_time'], 'message':k['message']})
  dfinf = pd.concat([dfinf, row.to_frame().T ], ignore_index=True)


TypeError: string indices must be integers, not 'str'

In [6]:
# check if it has the right content
dfinf.head(1)

,project,commit,author,time,message
0,ohdsi_patientlevelprediction,014d06ce8337aab2b4d7e9da90d2d0d22537d118,Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,parallelize getting blocks\n


In [7]:
#mode a means append, so you have all your projects in the same file
yournetid='econstan'
dfinf.to_csv(yournetid+'_project_summary.csv', index=False,sep=';', mode='a', header=False)

# Make sure you check in to your fork not just the notebook but also the csv files!!!

# Don't forget to add requested data from github and this notebook
### For each of the 10 projects go to their github repo and get the number of stars, number of forks, and the last commit date
### Report (in your notebook) the number of commits, the number of authors, and max and min time for each project based on WoC commits and also add the info you obtained from github

In [13]:
'''
Notes and Tips
what kind of source is worldofcode.org?
pay attention to how many commits you are getting, check for errors
you may need to retrieve commit sha1's first, then get commit content in batches
validation: does the number of commits or authors match that on GH?
WoC data is up to Jan 2025, so there may be some projects with later commits on GH

'''
import requests
# 1. loop through all project github repos

project_names = [
    "aidenlab/straw",
    "corybrunson/ggalluvial",
    "snowleopard/alga",
    "maximecb/gym-minigrid",
    "seung-lab/cloud-volume",
    "STVIR/pysot",
    "google-research/torchsde",
    "tidymodels/rsample",
    "coin-or/Ipopt",
    "OHDSI/PatientLevelPrediction",
]

github_links = [
    "https://github.com/aidenlab/straw",
    "https://github.com/corybrunson/ggalluvial",
    "https://github.com/snowleopard/alga",
    "https://github.com/maximecb/gym-minigrid",
    "https://github.com/seung-lab/cloud-volume",
    "https://github.com/STVIR/pysot",
    "https://github.com/google-research/torchsde",
    "https://github.com/tidymodels/rsample",
    "https://github.com/coin-or/Ipopt",
    "https://github.com/OHDSI/PatientLevelPrediction",
]

# 2. report # of stars, # forks, last commit date - Part 1 step 4 

session = requests.Session()
session.headers.update({"Accept": "application/vnd.github+json"})

project_data = []

for project in tqdm(projects):
    # Get stars, forks, and the current repository name
    response = session.get(
        f"https://api.github.com/repos/{project}",
        timeout=30,
    )
    response.raise_for_status()
    repo = response.json()
    response = session.get(
        f"https://api.github.com/repos/{repo['full_name']}/commits",
        params={"sha": repo["default_branch"], "per_page": 1},
        timeout=30,
    )
    response.raise_for_status()
    commits = response.json()

    project_data.append({
        "project": project,
        "github_url": repo["html_url"],
        "stars": repo["stargazers_count"],
        "forks": repo["forks_count"],
        "last_commit_date": (
            commits[0]["commit"]["committer"]["date"]
            if commits else None
        ),
    })

df_project_data = pd.DataFrame(project_data)

df_project_data.to_csv("df_project_data.csv", index=False)
df_project_data

# 3. report # of commits, # authors, max time, min time - Part 1 step 5

df_summary = (
    df_commit_data.groupby("project")
    .agg(
        number_of_commits=("commit", "nunique"),
        number_of_authors=("author", "nunique"),
        min_time=("committer_time", "min"),
        max_time=("committer_time", "max"),
    )
    .reset_index()
)

# the times aren't readable
for column in ["min_time", "max_time"]:
    df_summary[column] = pd.to_datetime(
        df_summary[column], unit="s", utc=True
    )

df_summary.to_csv("commits_authors_times.txt", sep="\t", index=False)
df_summary




 80%|████████  | 8/10 [00:00<00:00, 12.63it/s]


HTTPError: 403 Client Error: rate limit exceeded for url: https://api.github.com/repos/coin-or/Ipopt/commits?sha=stable%2F3.14&per_page=1

In [33]:

# 4. make CSV of projects information - econstan_project_summary.csv
# Part 1 step 6

# all the commits are saved to the csv called df_commit_data
# commit_data = pd.read_csv("df_commit_data.csv")
# print(commit_data[1])
# df_commit_data = pd.DataFrame(commit_data)
# v = commit_data[0]
# commit tree, [parent{s}], [author, unixtime, tz ], [ committer, unixtime, tz ], Commit_message ]
# print(v)
# print(commit_data)
df_commit_data.head(10)

new_row = []

# for project_name in df_commit_data['project'].unique():
# 	print(project_name)
for index, entry in df_commit_data.iterrows():
	# print(entry)
	new_row.append({
		"project_wocid": entry["project"],
		"commit_sha1": entry["sha1"],
		"author": entry["author"],
		"time": entry["committer_time"],
		"commit message": entry["message"]
	})

df_step_6 = pd.DataFrame(new_row)

df_step_6.to_csv("econstan_project_summary.csv", index=False)
df_step_6





,project_wocid,commit_sha1,author,time,commit message
0,aidenlab_straw,014d06ce8337aab2b4d7e9da90d2d0d22537d118,Muhammad Saad Shamim <sa501428@gmail.com>,1743867731,parallelize getting blocks\n
1,aidenlab_straw,02ef30c2693b1cba6fd1e5d2d694803aeed0403d,Muhammad Saad Shamim <sa501428@gmail.com>,1643839632,Merge pull request #91 from sa501428/helper3\n...
2,aidenlab_straw,0397a5027626eab94a326650cb7bf8bf73e6084c,nchernia <nchernia@gmail.com>,1476365702,updated README\n
3,aidenlab_straw,03cf1d6674f4d3eeed208f55bd5e31f74b381151,Muhammad S Shamim <sa501428@gmail.com>,1644528519,new usage\n
4,aidenlab_straw,03fbfdc1d863e8aed6ce70bee27c54daab2d388b,Dimos Goundaroulis <dgound@gmail.com>,1632171347,published v1.0.0.1 to pypi.org\n
...,...,...,...,...,...
22527,ohdsi_patientlevelprediction,ff93a358f46c27533d856442c575b7359d265364,egillax <egillax@gmail.com>,1682509725,remove docs folder\n
22528,ohdsi_patientlevelprediction,ffaec05960131a6324ac54662abad0dd3795038e,aniekmarkus <a.markus@erasmusmc.nl>,1677593502,Merge branch 'main' of https://github.com/mi-e...
22529,ohdsi_patientlevelprediction,ffc66ec7d1a2f7b93ea42bf66141161924aa8009,Jenna Reps <jreps@its.jnj.com>,1654113258,Update settings.R\n\n- updated shiny model set...
22530,ohdsi_patientlevelprediction,ffcdc0a064e22d24360dd9e54d25e17649de55eb,egillax <egillax@gmail.com>,1715761040,Deploying to gh-pages from @ OHDSI/PatientLeve...


In [48]:
# PART 2 ============================================================================


# Step 1 and 2
# month, # of commits
# Convert Unix timestamps to UTC dates
dates = pd.to_datetime(df_step_6["time"], unit="s", utc=True)
df_step_6["Month"] = dates.dt.strftime("%Y-%m")

monthly_commits = (
    df_step_6.groupby(["project_wocid", "Month"])["commit_sha1"]
    .nunique()
    .reset_index(name="#Commits")
)

# ADD optional section to fill in 0's where there are no commits for that month

filled_projects = []

for project_id, project_data in monthly_commits.groupby("project_wocid"):
    project_data = project_data.set_index("Month")

    # Include every month from this project's first to last month
    all_months = pd.period_range(
        start=project_data.index.min(),
        end=project_data.index.max(),
        freq="M",
    ).astype(str)

    project_data = project_data.reindex(all_months, fill_value='0')
    project_data.index.name = "Month"
    project_data["project_wocid"] = project_id

    filled_projects.append(project_data.reset_index())

monthly_commits = pd.concat(filled_projects, ignore_index=True)
monthly_commits = monthly_commits[
    ["project_wocid", "Month", "#Commits"]
]

# END optional section
monthly_commits["#Commits"] = monthly_commits["#Commits"].astype(int)
monthly_commits.to_csv("econstan_commits_timeseries.csv", index=False)
monthly_commits




# Step 3 and 4
# create line plot of the monthly data
# Plot Format:
# X-axis = months (YYYY-MM)
# Y-axis = number of commits
# Title = WOCProjectID
# Add axis labels and grid lines
# Save the plot as: netid_timeseries_<WOCProjectID>.png
# Export resolution: ≥300 DPI

import matplotlib.pyplot as plt

# Load CSV
# monthly = pd.read_csv("econstan_commits_timeseries.csv", sep=";")
for project_id, project_data in monthly_commits.groupby("project_wocid"):
	project_data = project_data.sort_values("Month")
      
	plt.figure(figsize=(10, 5))
	plt.plot(
		project_data["Month"],
		project_data["#Commits"],
		marker="o",
		linestyle="-",
	)

	plt.xlabel("Time (YYYY-MM)")
	plt.ylabel("Number of Commits")
	plt.title(f"Commit Activity: {project_id}")
	plt.grid(True)
	plt.xticks(rotation=45)
	plt.tight_layout()

	graph_filename = f"econstan_timeseries_{project_id}.png"
	plt.savefig(graph_filename, dpi=300)
	plt.close()




In [49]:
#  part 2 second section -- find the longest inactivity gap
project_mapping = {
    "aidenlab_straw": "aidenlab/straw",
    "corybrunson_ggalluvial": "corybrunson/ggalluvial",
    "snowleopard_alga": "snowleopard/alga",
    "maximecb_gym-minigrid": "maximecb/gym-minigrid",
    "seung-lab_cloud-volume": "seung-lab/cloud-volume",
    "stvir_pysot": "STVIR/pysot",
    "google-research_torchsde": "google-research/torchsde",
    "tidymodels_rsample": "tidymodels/rsample",
    "coin-or_ipopt": "coin-or/Ipopt",
    "ohdsi_patientlevelprediction": "OHDSI/PatientLevelPrediction",
}

columns = [
    "project",
    "number_of_commits",
    "number_of_authors",
    "max_time",
    "min_time",
    "number_of_stars",
    "number_of_forks",
    "last_commit_date",
    "LongestGapStart",
    "LongestGapEnd",
    "LongestGapLength",
	]

df_project_stats = pd.DataFrame(columns=columns)
github_stats = pd.read_csv("df_project_data.csv").set_index("project")

new_rows = []

# for each project, find the longest gap with no commits
for project_id, project_data in monthly_commits.groupby("project_wocid"):

    # get all data just for one project
    project_data = project_data.sort_values("Month")

    # track the max length of a gap
    longest_start = None
    longest_end = None
    longest_length = 0

    current_start = None
    current_length = 0


    # find the first month with no commits
    # count consecutive months with no commits
    for index, entry in project_data.iterrows():
        if entry["#Commits"] == 0:
            if current_length == 0:
                current_start = entry["Month"]

            current_length += 1

            if current_length > longest_length:
                longest_start = current_start
                longest_end = entry["Month"]
                longest_length = current_length
        else:
            current_start = None
            current_length = 0

    # Get commit records for this project
    commits = df_step_6[
        df_step_6["project_wocid"] == project_id
    ]

    times = pd.to_numeric(commits["time"], errors="raise")
    github_project = project_mapping[project_id]
    gh = github_stats.loc[github_project]

    # Values match the order of columns above
    new_rows.append([
        project_id,
        commits["commit_sha1"].nunique(),
        commits["author"].nunique(),
        pd.to_datetime(times.max(), unit="s", utc=True),
        pd.to_datetime(times.min(), unit="s", utc=True),
        gh["stars"],
        gh["forks"],
        gh["last_commit_date"],
        longest_start,
        longest_end,
        longest_length,
    ])


df_project_stats = pd.DataFrame(new_rows, columns=columns)
df_project_stats.to_csv("econstan_project_stats.csv", index=False)
df_project_stats




,project,number_of_commits,number_of_authors,max_time,min_time,number_of_stars,number_of_forks,last_commit_date,LongestGapStart,LongestGapEnd,LongestGapLength
0,aidenlab_straw,453,36,2025-06-29 21:21:26+00:00,2016-07-28 19:09:03+00:00,70,35,2025-04-06T02:23:28Z,2024-04,2025-03,12
1,coin-or_ipopt,8374,74,2026-03-09 19:56:22+00:00,2004-10-21 01:03:09+00:00,1790,318,2026-08-27T14:02:53Z,2017-07,2017-10,4
2,corybrunson_ggalluvial,864,14,2026-04-11 23:12:25+00:00,2016-05-16 21:24:54+00:00,524,34,2026-02-22T13:12:42Z,2023-08,2024-05,10
3,google-research_torchsde,615,23,2025-01-14 13:55:14+00:00,2020-07-06 23:58:04+00:00,1727,231,2024-12-30T10:52:51Z,2021-08,2023-06,23
4,maximecb_gym-minigrid,2115,193,2025-10-27 06:30:54+00:00,2017-12-13 17:26:52+00:00,2518,654,2026-09-10T21:43:25Z,2025-03,2025-05,3
5,ohdsi_patientlevelprediction,2679,63,2025-10-29 13:40:38+00:00,2015-03-22 19:08:53+00:00,222,96,2026-09-21T11:43:21Z,2019-09,2019-09,1
6,seung-lab_cloud-volume,3487,51,2025-10-31 19:38:23+00:00,2017-07-25 17:34:30+00:00,175,54,2026-09-28T22:12:01Z,2021-09,2021-09,1
7,snowleopard_alga,1997,66,2025-08-19 08:34:20+00:00,2016-12-05 00:30:03+00:00,763,74,2025-07-13T10:07:51Z,2023-05,2023-10,6
8,stvir_pysot,95,24,2025-06-22 13:38:05+00:00,2019-05-07 15:32:11+00:00,4603,1110,2025-06-22T03:59:22Z,2021-04,2023-03,24
9,tidymodels_rsample,1853,80,2026-04-14 13:53:36+00:00,2017-04-22 19:19:58+00:00,343,68,2026-09-17T17:57:29Z,2024-10,2025-02,5


In [54]:
# ADD NumCommitsAfterLastGap, NumberOfGapsInTimeline, ActivityPattern

import pandas as pd

stats = pd.read_csv("econstan_project_stats.csv")
stats["ActivityPattern"] = pd.Series(
    pd.NA, index=stats.index, dtype="string"
)

monthly = monthly_commits.copy()
monthly["#Commits"] = monthly["#Commits"].astype(int)
monthly["Month"] = pd.to_datetime(monthly["Month"], format="%Y-%m")

# add the activity pattern assessment
# steady, rising, declining, U-shaped, cyclical, irregular
activity_patterns = {
    "aidenlab_straw": "irregular",
    "corybrunson_ggalluvial": "declining",
    "snowleopard_alga": "declining",
    "maximecb_gym-minigrid": "U-shaped",
    "seung-lab_cloud-volume": "U-shaped",
    "stvir_pysot": "declining",
    "google-research_torchsde": "declining",
    "tidymodels_rsample": "inclining",
    "coin-or_ipopt": "steady",
    "ohdsi_patientlevelprediction": "inclining",
}

for index, row in stats.iterrows():
    project_id = row["project"]

    timeline = monthly[
        monthly["project_wocid"] == project_id
    ].sort_values("Month")

    # Count commits strictly after the longest gap's final month
    gap_end = row["LongestGapEnd"]

    if pd.notna(gap_end):
        gap_end = pd.to_datetime(gap_end, format="%Y-%m")
        commits_after = timeline.loc[
            timeline["Month"] > gap_end, "#Commits"
        ].sum()
    else:
        # No inactivity gap: use the entire timeline
        commits_after = timeline["#Commits"].sum()

    # Count separate runs of at least three inactive months
    number_of_gaps = 0
    inactive_months = 0

    for count in timeline["#Commits"]:
        if count == 0:
            inactive_months += 1

            # Count the run once, when it reaches three months
            if inactive_months == 3:
                number_of_gaps += 1
        else:
            inactive_months = 0

    stats.loc[index, "NumCommitsAfterLastGap"] = commits_after
    stats.loc[index, "NumberOfGapsInTimeline"] = number_of_gaps
    stats.loc[index, "ActivityPattern"] = str(activity_patterns[project_id])

stats[["NumCommitsAfterLastGap", "NumberOfGapsInTimeline"]] = (
    stats[["NumCommitsAfterLastGap", "NumberOfGapsInTimeline"]]
    .astype(int)
)

stats.to_csv("econstan_project_stats.csv", index=False)
stats

,project,number_of_commits,number_of_authors,max_time,min_time,number_of_stars,number_of_forks,last_commit_date,LongestGapStart,LongestGapEnd,LongestGapLength,NumCommitsAfterLastGap,NumberOfGapsInTimeline,ActivityPattern
0,aidenlab_straw,453,36,2025-06-29 21:21:26+00:00,2016-07-28 19:09:03+00:00,70,35,2025-04-06T02:23:28Z,2024-04,2025-03,12,27,5,irregular
1,coin-or_ipopt,8374,74,2026-03-09 19:56:22+00:00,2004-10-21 01:03:09+00:00,1790,318,2026-08-27T14:02:53Z,2017-07,2017-10,4,1310,2,steady
2,corybrunson_ggalluvial,864,14,2026-04-11 23:12:25+00:00,2016-05-16 21:24:54+00:00,524,34,2026-02-22T13:12:42Z,2023-08,2024-05,10,24,7,declining
3,google-research_torchsde,615,23,2025-01-14 13:55:14+00:00,2020-07-06 23:58:04+00:00,1727,231,2024-12-30T10:52:51Z,2021-08,2023-06,23,34,3,declining
4,maximecb_gym-minigrid,2115,193,2025-10-27 06:30:54+00:00,2017-12-13 17:26:52+00:00,2518,654,2026-09-10T21:43:25Z,2025-03,2025-05,3,19,1,U-shaped
5,ohdsi_patientlevelprediction,2679,63,2025-10-29 13:40:38+00:00,2015-03-22 19:08:53+00:00,222,96,2026-09-21T11:43:21Z,2019-09,2019-09,1,1856,0,inclining
6,seung-lab_cloud-volume,3487,51,2025-10-31 19:38:23+00:00,2017-07-25 17:34:30+00:00,175,54,2026-09-28T22:12:01Z,2021-09,2021-09,1,1009,0,U-shaped
7,snowleopard_alga,1997,66,2025-08-19 08:34:20+00:00,2016-12-05 00:30:03+00:00,763,74,2025-07-13T10:07:51Z,2023-05,2023-10,6,53,5,declining
8,stvir_pysot,95,24,2025-06-22 13:38:05+00:00,2019-05-07 15:32:11+00:00,4603,1110,2025-06-22T03:59:22Z,2021-04,2023-03,24,14,5,declining
9,tidymodels_rsample,1853,80,2026-04-14 13:53:36+00:00,2017-04-22 19:19:58+00:00,343,68,2026-09-17T17:57:29Z,2024-10,2025-02,5,156,1,inclining


## Interpretation

- add an explanation for the trends you noted. See the definitions below. Also specify the number of gaps (of at least three months of inactivity) are there for each project.
- example Trends:

- - steady, rising, declining, U-shaped: Long-term, general movements in activity levels. An upward trend might indicate that a person is becoming more active over several months, while a downward trend could signal a decrease in mobility.

- - Seasonal/Cyclical patterns: Predictable and repeated patterns of activity that occur at fixed intervals, such as quarterly or annually.

- - Irregular or random variations: Unpredictable changes in activity that cannot be explained by trends or seasonal patterns.


